# 1. Mount gdrive and import libs

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')
import pandas as pd
import numpy as np
import re
from pathlib import Path
from typing import Literal
import matplotlib.pyplot as plt

from skimage.filters import gaussian, threshold_otsu
from skimage.segmentation import *
from skimage.color import rgb2gray, rgb2hsv
import skimage.io as skio
from skimage.feature import graycomatrix, graycoprops
import skimage.morphology as skmor
import skimage.measure as skmea
from skimage.morphology import (
	disk,
	binary_opening,
	binary_closing,
	remove_small_objects,
	remove_small_holes
)

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, Subset

!pip install xgboost
from sklearn.metrics import accuracy_score,classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

Mounted at /content/gdrive


# Unzip dataset zip file

In [ ]:
#!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8"
#!unzip "/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large.zip" -d "/content/gdrive/MyDrive/BMET5933/WEEK_8"

Archive:  /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img.zip
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/027.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/027_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/032.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/032_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/034.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/034_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/071.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/071_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/076.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/076_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/082.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img/082_Mask.png  
 extracting: /content/gdrive/MyDrive/BMET5933/WEEK_8/hep

# Dataset class definition:
Chatgpt5.4 is used for designing, debugging and adding boundary condition structure of this dataset class.

In [ ]:
class Hep2Dataset(Dataset):
	def __init__(self, dataset_root, csv_path, transform=None):
		"""
		Initialize the dataset.

		Args:
			dataset_root (str or Path): Root folder of the image dataset.
			csv_path (str or Path): CSV file containing image file names and labels.
			transform (callable, optional): Transform to apply to each image.
		"""
		self.dataset_root = Path(dataset_root)
		self.csv_path = Path(csv_path)
		self.transform = transform

		# Read all samples once and store them as a unified list
		self.samples = self.read_label(self.csv_path)

	def read_label(self, csv_path):
		"""
		Read the CSV file and build a list of (image_path, label).

		Args:
			csv_path (Path): Path to the CSV label file.

		Returns:
			list: A list of tuples, where each tuple is (image_path, label).
		"""
		df = pd.read_csv(csv_path)

		samples = []
		for _, row in df.iterrows():
			img_name = row["file"]
			label = int(row["class"])  # Convert label to integer
			mask_path = self.dataset_root / f'{img_name}_Mask.png'
			img_path = self.dataset_root / f'{img_name}.png'
			samples.append((img_path, mask_path, label))

		return samples

	def __len__(self):
		"""
		Return the total number of samples in the dataset.
		"""
		return len(self.samples)

	def __getitem__(self, idx):
		"""
		Get one sample by index.

		Args:
			idx (int): Index of the sample.

		Returns:
			tuple: (image, mask, label)
		"""
		img_path, mask_path, label = self.samples[idx]
		img = skio.imread(img_path)
		mask = skio.imread(mask_path)

		if self.transform is not None:
			img = self.transform(img)

		return img, mask, label

# Shape feature extractor:
1.Chatgpt5.4 used for designing shape feature extraction class.

2.Here I used circularity, solidity and eccentricity beacuse we don't have metadata like what we have in DICOM format. So some features like perimeter and area, we cannnot get.

In [ ]:
class ShapeFeatureExtractor:
	def __init__(self):
		return 

	def _get_largest_region(self, mask):
		"""
		Get the largest connected component from a mask.

		Args:
			mask (ndarray): Input mask.

		Returns:
			regionprops object or None: Largest region if exists, else None.
		"""
		mask = mask > 0
		labeled_mask = skmea.label(mask)

		regions = skmea.regionprops(labeled_mask)
		if len(regions) == 0:
			return None

		largest_region = max(regions, key=lambda x: x.area)
		return largest_region

	def get_solidity(self, mask):
		"""
		Calculate solidity of the largest connected component.

		Args:
			mask (ndarray): Input mask.

		Returns:
			float: Solidity value. Returns 0.0 if no valid region exists.
		"""
		largest_region = self._get_largest_region(mask)
		return self._get_solidity_from_region(largest_region)

	def get_eccentricity(self, mask):
		"""
		Calculate eccentricity of the largest connected component.

		Args:
			mask (ndarray): Input mask.

		Returns:
			float: Eccentricity value. Returns 0.0 if no valid region exists.
		"""
		largest_region = self._get_largest_region(mask)
		return self._get_eccentricity_from_region(largest_region)

	def get_circularity(self, mask):
		"""
		Calculate circularity of the largest connected component.

		Formula:
			circularity = 4 * pi * area / perimeter^2

		Args:
			mask (ndarray): Input mask.

		Returns:
			float: Circularity value. Returns 0.0 if no valid region exists
				   or perimeter is zero.
		"""
		largest_region = self._get_largest_region(mask)
		return self._get_circularity_from_region(largest_region)

	def _get_solidity_from_region(self, region):
		"""
		Calculate solidity from a regionprops object.

		Args:
			region: regionprops object or None.
		Returns:
			float: Solidity value. Returns 0.0 if no valid region exists.
		"""
		if region is None:
			return 0.0

		return float(region.solidity)

	def _get_eccentricity_from_region(self, region):
		"""
		Calculate eccentricity from a regionprops object.

		Args:
			region: regionprops object or None.
		Returns:
			float: Eccentricity value. Returns 0.0 if no valid region exists.
		"""
		if region is None:
			return 0.0

		return float(region.eccentricity)

	def _get_circularity_from_region(self, region):
		"""
		Calculate circularity from a regionprops object.

		Formula:
			circularity = 4 * pi * area / perimeter^2

		Args:
			region: regionprops object or None.
		Returns:
			float: Circularity value. Returns 0.0 if no valid region exists
				   or perimeter is zero.
		"""
		if region is None:
			return 0.0

		area = region.area
		perimeter = region.perimeter

		if perimeter == 0:
			return 0.0

		circularity = 4.0 * np.pi * area / (perimeter ** 2)
		return float(circularity)


	def get_shape_features(self, mask):
		"""
		Extract all shape features from a single mask.

		Args:
			mask (ndarray): Input mask.

		Returns:
			dict: Dictionary containing shape features.
		"""
		largest_region = self._get_largest_region(mask)
		single_feature = {
			"circularity": self._get_circularity_from_region(largest_region),
			"solidity": self._get_solidity_from_region(largest_region),
			"eccentricity": self._get_eccentricity_from_region(largest_region),
		}
		return single_feature
	
	def get_shape_features_list(self, masks):
		"""
		Get a list of shape features for a list of masks.

		Args:
			masks (list): List of input masks.
		Returns:
			list: List of dictionaries containing shape features for each mask.
		"""
		shape_feature_list = [self.get_shape_features(mask) for mask in masks]
		return shape_feature_list
	

# GLCM feature exatractor:

In [ ]:
class GlcmFeatureExtractor:
	def __init__(self):
		"""
		Initialize the GLCM feature extractor.

		Args:
			image (ndarray): Input image.
			label (int): Corresponding label.
			mask (ndarray): Corresponding mask.
		"""

		# glcm configuration
		self.distance = 1
		self.angle = 0
		self.levels = 32
		self.symmetric = False
		self.normed = False

	def get_gray_image(self, image):
		"""
		Get the gray-level image.

		Args:
			image (ndarray): Input image.
		Returns:
			ndarray: Gray-level image.
		"""
		gray_image = rgb2gray(image)
		quantized_gray_image = np.floor(gray_image * (self.levels - 1)).astype(np.uint8)
		return quantized_gray_image
	
	def get_masked_gray_image(self, image, mask):
		"""
		Get the gray-level image with the background masked out.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			ndarray: Gray-level image with background masked out.
		"""
		background_mask  = ~(mask > 0)
		gray_image = self.get_gray_image(image)

		# only keep foreground region
		masked_image = gray_image.copy()
		masked_image[background_mask] = 0
		return masked_image
	
	def get_glcm_features(self, image, mask):
		glcm_features = {}

		# get masked gray image first
		masked_image = self.get_masked_gray_image(image, mask)

		# get GLCM matrix
		glcm = graycomatrix(
					masked_image.astype(np.uint8), 
					distances=[self.distance], 
					angles=[self.angle], 
					levels=self.levels, 
					symmetric=self.symmetric, 
					normed=self.normed
					)
		# extract GLCM features
		glcm_contrast = graycoprops(glcm, prop='contrast')[0, 0]
		glcm_dissimilarity = graycoprops(glcm, prop='dissimilarity')[0, 0]
		glcm_homogeneity = graycoprops(glcm, prop='homogeneity')[0, 0]
		glcm_energy = graycoprops(glcm, prop='energy')[0, 0]
		glcm_correlation = graycoprops(glcm, prop='correlation')[0, 0]

		# store features in a dictionary
		glcm_features = {
			"contrast": glcm_contrast,
			"dissimilarity": glcm_dissimilarity,
			"homogeneity": glcm_homogeneity,
			"energy": glcm_energy,
			"correlation": glcm_correlation
		}

		return glcm_features
	
	def get_glcm_feature_list(self, images, masks):
		"""
		Get a list of GLCM features for a list of masked images.

		Args:
			images (list): List of input images.
			masks (list): List of input masks.
		Returns:
			list: List of dictionaries containing GLCM features for each masked image.
		"""
		glcm_feature_list = []
		for image, mask in zip(images, masks):
			glcm_features_dict = self._get_glcm_features(image, mask) # get dictionary of GLCM features
			glcm_feature_list.append(glcm_features_dict) # store as list
		return glcm_feature_list


# Color feature extractor:

In [ ]:
class ColorFeatureExtractor:
	def __init__(self):
		return
	
	def get_masked_color_image(self, image, mask):
		"""
		Get the color image with the background masked out.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			ndarray: Color image with background masked out.
		"""
		background_mask  = ~(mask > 0)
		masked_image = image.copy()
		masked_image[background_mask] = 0
		return masked_image
	
	def get_max_min_rgb(self, image, mask):
		"""
		Calculate the max and min pixel intensities in the masked image for each color channel.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			tuple: (max_rgb, min_rgb) where each is a list of max and min for each color channel.
		"""
		masked_image = self.get_masked_color_image(image, mask)
		max_rgb = masked_image[mask > 0].max(axis=0)
		min_rgb = masked_image[mask > 0].min(axis=0)
		return max_rgb, min_rgb
	
	def get_rgb_avg(self, image, mask):
		"""
		Calculate the average pixel intensities in the masked image for each color channel.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			tuple: (avg_rgb) where avg_rgb is a list of average intensities for each color channel.
		"""
		masked_image = self.get_masked_color_image(image, mask)
		avg_rgb = masked_image[mask > 0].mean(axis=0)
		return avg_rgb
	
	def get_rgb_std(self, image, mask):
		"""
		Calculate the standard deviation of pixel intensities in the masked image for each color channel.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			tuple: (std_rgb) where std_rgb is a list of standard deviations for each color channel.
		"""
		masked_image = self.get_masked_color_image(image, mask)
		std_rgb = masked_image[mask > 0].std(axis=0)
		return std_rgb
	
	def get_rgb_histogram(self, image, mask):
		"""
		Calculate the mean and standard deviation of pixel intensities in the masked image.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			tuple: (mean_intensity, std_intensity) where each is a list of mean and std for each color channel.
		"""
		masked_image = self.get_masked_color_image(image, mask)
		hists = {}
		channels = ['R', 'G', 'B']
		for channel in range(masked_image.shape[2]):
			channel_data = masked_image[:, :, channel]
			channel_histogram, bin_edges = np.histogram(channel_data[mask > 0], bins=256, range=(0, 256))
			hists[channels[channel]] = channel_histogram
		return hists, bin_edges
	
	def get_gray_histogram(self, image, mask):
		"""
		Calculate the grayscale histogram of pixel intensities in the masked image.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			tuple: (histogram, bin_edges) where histogram is the grayscale histogram and bin_edges are the bin edges.
		"""
		masked_image = self.get_masked_color_image(image, mask)
		gray_image = rgb2gray(masked_image)
		histogram, bin_edges = np.histogram(gray_image[mask > 0], bins=256, range=(0, 1))
		return histogram, bin_edges	
	
	def _get_rgb_histogram_list(self, images, masks):
		"""
		Calculate the RGB histograms of pixel intensities for a list of masked images.

		Args:
			images (list): List of input images.
			masks (list): List of input masks.
		Returns:
			tuple: (rgb_hists, bin_edges) where rgb_hists is a list of RGB histograms for each masked image and bin_edges are the bin edges.
		"""
		rgb_hists = []
		for image, mask in zip(images, masks):
			hists, bin_edges = self.get_rgb_histogram(image, mask)
			rgb_hists.append(hists)
		return rgb_hists, bin_edges
	
	def _get_gray_histogram_list(self, images, masks):
		"""
		Calculate the grayscale histograms of pixel intensities for a list of masked images.

		Args:
			images (list): List of input images.
			masks (list): List of input masks.
		Returns:
			list: A list of tuples, where each tuple is (histogram, bin_edges) for the grayscale histogram of a masked image.
		"""
		gray_hists = []
		for image, mask in zip(images, masks):
			histogram, bin_edges = self.get_gray_histogram(image, mask)
			gray_hists.append((histogram, bin_edges))
		return gray_hists, bin_edges
	
	def get_rgb_avg_list(self, images, masks):
		"""
		Calculate the average RGB pixel intensities for a list of masked images.

		Args:
			images (list): List of input images.
			masks (list): List of input masks.
		Returns:
			list: A list of average RGB values for each masked image.
		"""
		avg_rgb_list = []
		for image, mask in zip(images, masks):
			avg_rgb = self.get_rgb_avg(image, mask)
			avg_rgb_list.append(avg_rgb)
		return avg_rgb_list
	
	def get_rgb_std_list(self, images, masks):
		"""
		Calculate the standard deviation of RGB pixel intensities for a list of masked images.

		Args:
			images (list): List of input images.
			masks (list): List of input masks.
		Returns:
			list: A list of RGB standard deviation values for each masked image.
		"""
		std_rgb_list = []
		for image, mask in zip(images, masks):
			std_rgb = self.get_rgb_std(image, mask)
			std_rgb_list.append(std_rgb)
		return std_rgb_list
	
	def get_rgb_max_min_list(self, images, masks):
		"""
		Calculate the max and min RGB pixel intensities for a list of masked images.

		Args:
			images (list): List of input images.
			masks (list): List of input masks.
		Returns:
			tuple: (max_rgb_list, min_rgb_list) where each is a list of max and min RGB values for each masked image.
		"""
		max_rgb_list = []
		min_rgb_list = []
		for image, mask in zip(images, masks):
			max_rgb, min_rgb = self.get_rgb_max_min(image, mask)
			max_rgb_list.append(max_rgb)
			min_rgb_list.append(min_rgb)
		return max_rgb_list, min_rgb_list
	
	def get_rgb_color_features(self, image, mask):
		"""
		Calculate a set of RGB color features for a masked image.

		Args:
			image (ndarray): Input image.
			mask (ndarray): Input mask.
		Returns:
			dict: A dictionary containing RGB color features such as average, std, max, and min for each color channel.
		"""
		color_features = {}
		color_features["avg_rgb"] = self.get_rgb_avg(image, mask)
		color_features["std_rgb"] = self.get_rgb_std(image, mask)
		color_features["max_rgb"], color_features["min_rgb"] = self.get_rgb_max_min(image, mask)
		return color_features


In [ ]:
train_dataset = Hep2Dataset(
	dataset_root="/content/gdrive/MyDrive/BMET5933/WEEK_8/hep2img_large",
	csv_path="/content/gdrive/MyDrive/BMET5933/WEEK_8/labels_large.csv"
)
train_test_split_ratio = 0.2
train_indices, test_indices = train_test_split(range(len(train_dataset)), test_size = train_test_split_ratio, random_state=42)
train_subset = Subset(train_dataset, train_indices)
test_subset = Subset(train_dataset, test_indices)

In [ ]:
shape_extractor = ShapeFeatureExtractor()
color_extractor = ColorFeatureExtractor()
glcm_extractor = GlcmFeatureExtractor()

feature_list = []
for i in range(len(train_dataset)):
	feature = {}
	feature.update(shape_extractor.get_shape_features(train_dataset[i]))
	feature.update(color_extractor.get_rgb_color_features(train_dataset[i]))
	feature.update(glcm_extractor.get_glcm_features(train_dataset[i]))
	feature['label'] = train_dataset[i][2]
	feature_list.append(feature)
	
feature_df = pd.DataFrame(feature_list)
feature_df.head()

# Classifier

In [ ]:
# These are for exporting the notebook as a PDF later if needed
!apt-get update
!sudo apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic pandoc
!jupyter nbconvert --to pdf /content/gdrive/MyDrive/Colab_Notebooks/BMET5933/WEEK_8/Week_8_9_Last_name_Code_file_ipynb_draft_version.ipynb

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
pandoc is already the newest version (2.9.2.1-3ubuntu2).
texlive-fonts-recommended is already 